# Sesión 3.06 · LangChain sobre colecciones existentes

Hasta aquí hemos utilizado los SDKs nativos de los motores. Esa repetición no era un accidente: antes de introducir una abstracción conviene saber qué operaciones, parámetros y garantías estamos dejando fuera de la interfaz.

En este notebook conectaremos LangChain a las colecciones de Chroma y Qdrant que ya poblamos en sus laboratorios. No repetiremos los 50.000 upsert y tampoco incorporaremos un LLM. La pregunta no es si LangChain permite escribir menos líneas, sino qué contrato común aporta y en qué puntos vuelven a aparecer las decisiones específicas de cada proveedor.

Si una colección no está disponible, el notebook registrará la causa. Una lista vacía puede ser un resultado de búsqueda correcto; no puede utilizarse para fingir que una conexión fallida simplemente no encontró documentos.

<a id="s03-langchain-indice"></a>

## Índice de contenidos

1. [Embeddings](#s03-langchain-embeddings)
2. [Conexión](#s03-langchain-conexion)
3. [Document](#s03-langchain-document)
4. [Búsqueda por similitud](#s03-langchain-busqueda)
5. [Retrievers](#s03-langchain-retrievers)
6. [MMR](#s03-langchain-mmr)
7. [Filtros](#s03-langchain-filtros)
8. [Operaciones CRUD](#s03-langchain-crud)
9. [Cadena de recuperación](#s03-langchain-cadena)
10. [Comparación con SDK nativo](#s03-langchain-comparacion)
11. [Límites de la abstracción](#s03-langchain-limites)
12. [Próximos pasos](#s03-langchain-proximos-pasos)



In [ ]:
from pathlib import Path
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vector_database_session import (
    ProviderRun,
    SearchHit,
    evaluate_run,
    exact_top_k,
    iter_record_batches,
    load_session_data,
    record_id_for_product,
    validate_resource_name,
    wait_until,
    write_provider_run,
)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env")
data = load_session_data(memory_map=True)
TELEVISOR_QUERY_ID = "semantic-101352"
TALADRO_QUERY_ID = "semantic-100455"
TOP_K = 10

In [ ]:
import chromadb
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

CHROMA_COLLECTION = validate_resource_name(os.getenv("CHROMA_COLLECTION", "bbdd-vectoriales-s03-chroma"))
QDRANT_COLLECTION = validate_resource_name(os.getenv("QDRANT_COLLECTION", "bbdd-vectoriales-s03-qdrant"))

<a id="s03-langchain-embeddings"></a>

## 1. Embeddings

LangChain no elimina el contrato del embedding. La colección existente contiene vectores producidos con multilingual-e5-small, normalización L2 y prefijos distintos para documentos y consultas. Si aquí utilizáramos otra configuración, seguiríamos obteniendo vectores de 384 dimensiones, pero estaríamos consultando una geometría diferente.

Por eso configuramos passage para documentos nuevos, query para consultas y normalización para ambos casos. En este notebook solo volveremos a codificar las consultas y el documento temporal de prueba; los 50.000 productos ya están almacenados con sus vectores originales.

La abstracción facilita la conexión con un vector store, pero no puede corregir un contrato de entrada mal aplicado. Antes de comparar Chroma, Qdrant o LangChain, debemos comprobar que todos están mirando el mismo espacio.

La separación entre documentos y consultas no es un detalle de sintaxis. Durante el entrenamiento, E5 aprendió que un texto de documento y una necesidad de búsqueda cumplen papeles diferentes. Si quitamos esos prefijos, seguimos produciendo vectores válidos, pero modificamos su posición en el espacio y, con ello, el ranking esperado.

Esta comprobación mantiene aislada la variable que queremos estudiar. Antes de atribuir una diferencia a LangChain, debemos asegurarnos de que todos los caminos están consultando el mismo espacio vectorial.


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)
television_text = str(data.query_row(TELEVISOR_QUERY_ID)["query_text"])
encoded_query = np.asarray(embeddings.embed_query(television_text), dtype=np.float32)
reference_query = data.query_vector(TELEVISOR_QUERY_ID)
print({
    "shape": encoded_query.shape,
    "norm": float(np.linalg.norm(encoded_query)),
    "cosine_with_snapshot": float(encoded_query @ reference_query),
})

La consulta recalculada debería quedar muy cerca del embedding que aparece en el snapshot. Puede haber diferencias pequeñas por versión del backend o por detalles numéricos, pero una diferencia grande obliga a detenerse.

Si cambiamos el embedding y continuamos, cualquier variación posterior podría atribuirse erróneamente a LangChain cuando en realidad estaríamos comparando dos consultas diferentes. Esta comprobación no es una formalidad: mantiene aislada la variable que queremos observar.


<a id="s03-langchain-conexion"></a>

## 2. Conexión

Cada proveedor puede encontrarse en dos estados distintos: disponible o no disponible. La colección debe existir porque la creó el laboratorio nativo; usar una operación de crear-si-no-existe aquí sería un error de diseño. Una errata en el nombre podría crear una colección vacía y hacer que una búsqueda sin resultados parezca válida.

El preflight comprueba la conexión, la existencia de la colección y el formato esperado. Si uno de esos pasos falla, el notebook conserva la excepción y explica por qué ese proveedor no participa en la comparación.

Esta distinción evita un problema frecuente: confundir una lista vacía devuelta por un buscador con la ausencia total de un buscador operativo.

La disponibilidad debe conservarse como parte del resultado de la prueba. En una aplicación real podríamos reintentar o degradar el servicio. En el notebook necesitamos algo más básico: distinguir entre un proveedor que respondió con cero documentos y un proveedor al que nunca llegamos a consultar.

Esa distinción evita una comparación injusta. La ausencia de una conexión no es evidencia de un ranking vacío.


In [ ]:
stores = {}
native_clients = {}
unavailable = {}

try:
    chroma_client = chromadb.HttpClient(
        host=os.getenv("CHROMA_HOST", "localhost"),
        port=int(os.getenv("CHROMA_PORT", "8000")),
    )
    chroma_native = chroma_client.get_collection(CHROMA_COLLECTION, embedding_function=None)
    if chroma_native.count() != 50_000:
        raise RuntimeError(f"Chroma contiene {chroma_native.count()} registros; ejecuta primero 03_02")
    stores["chroma"] = Chroma(
        client=chroma_client,
        collection_name=CHROMA_COLLECTION,
        embedding_function=embeddings,
    )
    native_clients["chroma"] = chroma_native
except Exception as error:
    unavailable["chroma"] = f"{type(error).__name__}: {error}"

try:
    qdrant_client = QdrantClient(
        url=os.getenv("QDRANT_URL", "http://localhost:6333"),
        api_key=os.getenv("QDRANT_API_KEY", "").strip() or None,
    )
    qdrant_info = qdrant_client.get_collection(QDRANT_COLLECTION)
    if qdrant_info.points_count != 50_000:
        raise RuntimeError(f"Qdrant contiene {qdrant_info.points_count} registros; ejecuta primero 03_05")
    stores["qdrant"] = QdrantVectorStore(
        client=qdrant_client,
        collection_name=QDRANT_COLLECTION,
        embedding=embeddings,
        content_payload_key="text",
        metadata_payload_key="metadata",
    )
    native_clients["qdrant"] = qdrant_client
except Exception as error:
    unavailable["qdrant"] = f"{type(error).__name__}: {error}"

print({"available": sorted(stores), "unavailable": unavailable})
if not stores:
    raise RuntimeError(f"Ningún proveedor disponible: {unavailable}")

<a id="s03-langchain-document"></a>

## 3. Document

Document es la unidad con la que LangChain transporta texto y metadatos entre componentes. No necesita que LangChain haya ejecutado la búsqueda. Podemos convertir directamente una respuesta nativa de Chroma o Qdrant en documentos cuando otra capa ya ha recuperado los vecinos.

Esta primera conversión nos permite separar dos responsabilidades. El SDK nativo sigue siendo quien consulta la base y entrega sus campos; LangChain recibe una representación común que puede circular por un retriever o una cadena.

No debemos interpretar esa forma común como una pérdida de identidad. Decidir qué texto, ID y metadatos llegan a Document sigue siendo responsabilidad de nuestro adaptador.

Esta conversión permite observar qué gana y qué pierde la abstracción. Ganamos una estructura que otras piezas de LangChain entienden sin conocer Chroma ni Qdrant. Pero no debemos perder el ID ni los metadatos que permiten volver desde un fragmento de contexto hasta el producto y la operación que lo recuperó.

En un sistema real ese rastro es indispensable. Si un usuario pregunta de dónde salió una afirmación, el sistema debe poder señalar el documento e identidad que hicieron posible recuperarla.


In [ ]:
exact_example = exact_top_k(data, reference_query, k=3)
documents_from_native = [
    Document(
        id=hit.record_id,
        page_content=hit.title,
        metadata={
            "record_id": hit.record_id,
            "product_id": hit.product_id,
            "brand": hit.brand,
            "native_score": hit.native_score,
            "score_kind": hit.score_kind,
        },
    )
    for hit in exact_example
]
documents_from_native

La clase Document no normaliza los scores ni garantiza que su ID coincida automáticamente con la clave primaria del proveedor. Si necesitamos comparar rankings, rastrear el origen de un fragmento o borrar un documento concreto, debemos conservar explícitamente esa identidad.

Esta es una frontera importante de la abstracción. La interfaz común nos ayuda a componer resultados, pero no decide qué evidencia nativa es necesaria para interpretar esos resultados.


<a id="s03-langchain-busqueda"></a>

## 4. Búsqueda por similitud

Ahora pediremos a cada vector store una búsqueda por similitud y una búsqueda con score. LangChain ofrece nombres de método parecidos para ambos proveedores, pero no convierte sus puntuaciones en una escala universal.

Una distancia devuelta por Chroma y una similitud devuelta por Qdrant pueden inducir rankings equivalentes y, aun así, representar cantidades distintas. Por eso seguiremos tratando el score como una propiedad del proveedor, no como un número que pueda compararse directamente entre motores.

La utilidad de esta capa es reducir la ceremonia de la consulta. Su límite es que no debe borrar la semántica de la respuesta.

La palabra similarity no debe hacernos olvidar que cada proveedor materializa la búsqueda de una manera distinta. Un motor puede devolver distancia, otro similitud y otro una transformación interna. LangChain simplifica la llamada, pero no reescribe la geometría ni el contrato del score.

Por eso, cuando el objetivo sea evaluar recall o explicar una posición del ranking, seguiremos conservando IDs y scores nativos. La interfaz común sirve para construir; la evidencia nativa sirve para interpretar.


In [ ]:
scored_results = {}
for provider, store in stores.items():
    scored_results[provider] = store.similarity_search_with_score(television_text, k=5)
    print(f"\n{provider.upper()}")
    for rank, (document, score) in enumerate(scored_results[provider], start=1):
        print(rank, document.metadata.get("record_id"), float(score), document.page_content[:90])

<a id="s03-langchain-retrievers"></a>

## 5. Retrievers

Un retriever reduce el contrato a una transformación sencilla: recibe una consulta de texto y devuelve una lista de documentos. Esta interfaz resulta muy útil cuando queremos combinar recuperación, formateo de contexto y, más adelante, generación.

A cambio, parte de la información nativa deja de estar en primer plano. El retriever no administra colecciones, no configura índices, no explica la consistencia de una escritura y no sustituye el diagnóstico de una petición fallida.

La decisión no es elegir entre retriever y SDK nativo. Podemos utilizar el primero para componer una aplicación y conservar el segundo cuando necesitamos administrar, observar o depurar el motor.

El retriever desacopla al consumidor del contexto de la base concreta. Una cadena posterior no necesita saber si los documentos vienen de Chroma, Qdrant o una lista precalculada. Esta propiedad facilita los prototipos y permite intercambiar estrategias con menos cambios aguas abajo.

Pero un contrato pequeño no puede expresar todas las garantías. El retriever no puede prometer qué índice se usó, cómo se aplicó un filtro o qué consistencia observó la lectura. Esas preguntas no desaparecen: quedan fuera de la interfaz.


In [ ]:
retrievers = {
    provider: store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
    for provider, store in stores.items()
}
retrieved = {
    provider: retriever.invoke(television_text)
    for provider, retriever in retrievers.items()
}
{provider: [document.metadata.get("record_id") for document in documents] for provider, documents in retrieved.items()}

<a id="s03-langchain-mmr"></a>

## 6. MMR

Maximal Marginal Relevance introduce diversidad después de recuperar candidatos. LangChain solicita primero un conjunto más amplio y selecciona documentos que equilibran similitud con la consulta y novedad frente a los documentos ya elegidos.

MMR no cambia el embedding, no modifica el índice y no convierte una búsqueda aproximada en exacta. Fetch_k define el universo sobre el que puede actuar: un documento que no aparezca entre esos candidatos no puede volver a entrar por el hecho de ser diverso.

Por tanto, MMR cambia el objetivo del ranking. Ya no buscamos únicamente los vecinos más próximos, sino un conjunto que también reduzca la redundancia.

MMR no es un reranker ni vuelve a leer los documentos con otro modelo. Opera sobre los candidatos ya recuperados y modifica la selección final para reducir redundancia. Por eso no mejora automáticamente la relevancia: cambia el objetivo del ranking.

El valor de fetch_k determina el conjunto sobre el que puede actuar. Un documento que no aparece entre esos candidatos no puede volver a entrar por ser diverso. Elegirlo implica decidir cuánto trabajo adicional aceptamos antes de seleccionar el resultado final.


In [ ]:
mmr_results = {}
for provider, store in stores.items():
    mmr_results[provider] = store.max_marginal_relevance_search(
        television_text, k=5, fetch_k=30, lambda_mult=0.65
    )
{provider: [document.metadata.get("product_id") for document in documents] for provider, documents in mmr_results.items()}

Si MMR elimina duplicados pero introduce productos menos relevantes, no ha fallado necesariamente. Hemos pedido una transformación que valora la diversidad además de la similitud.

La calidad de esa decisión debe evaluarse con una pregunta distinta a recall frente al oráculo exacto. Hay que decidir cuánta diversidad necesita el usuario y qué pérdida de similitud resulta aceptable para conseguirla.


<a id="s03-langchain-filtros"></a>

## 7. Filtros

Los filtros muestran rápidamente dónde se rompe la neutralidad aparente. Chroma acepta un diccionario de metadatos; Qdrant necesita un objeto Filter y una ruta física dentro del payload. El método de alto nivel puede tener el mismo nombre, pero el argumento sigue perteneciendo al proveedor.

Esta diferencia no es un detalle incómodo de la API. Expresa cómo cada motor modela sus datos y construye sus índices auxiliares. Ocultarla por completo impediría diagnosticar por qué un filtro devuelve pocos resultados, tarda demasiado o no puede expresarse.

LangChain puede ayudarnos a componer el resultado filtrado, pero el contrato del filtro debe seguir conociéndose y verificándose en la capa nativa.

El filtro de marca muestra con claridad el límite de la neutralidad. Para LangChain es un argumento que acompaña a la búsqueda; para cada proveedor es una expresión que debe traducirse a su modelo de metadata o payload y a sus estructuras de filtrado.

Aceptar un objeto propio del proveedor no es un fallo de la abstracción. Es la forma honesta de no ocultar diferencias que afectan a corrección, rendimiento y seguridad.


In [ ]:
drill_text = str(data.query_row(TALADRO_QUERY_ID)["query_text"])
filtered_documents = {}
if "chroma" in stores:
    filtered_documents["chroma"] = stores["chroma"].similarity_search(
        drill_text, k=10, filter={"brand": "Einhell"}
    )
if "qdrant" in stores:
    qdrant_filter = models.Filter(
        must=[models.FieldCondition(
            key="metadata.brand",
            match=models.MatchValue(value="Einhell"),
        )]
    )
    filtered_documents["qdrant"] = stores["qdrant"].similarity_search(
        drill_text, k=10, filter=qdrant_filter
    )
for provider, documents in filtered_documents.items():
    assert documents and all(document.metadata["brand"] == "Einhell" for document in documents)
{provider: [document.metadata["product_id"] for document in documents] for provider, documents in filtered_documents.items()}

Un retriever configurable puede encapsular parte de esa lógica, pero no convierte las capacidades de los motores en idénticas. Rangos, arrays anidados, texto completo, geolocalización o aislamiento por tenant vuelven a exponer la gramática del backend.

La neutralidad útil está en poder componer documentos y estrategias de recuperación. No consiste en negar las diferencias que afectan a la corrección, el rendimiento o la operación.


<a id="s03-langchain-crud"></a>

## 8. Operaciones CRUD

Insertaremos un único documento temporal de prueba mediante cada vector store. No volveremos a cargar el catálogo ni crearemos una colección paralela: la prueba debe demostrar qué añade LangChain sobre el mismo sistema que ya consultamos con el SDK nativo.

Usaremos el UUIDv5 reservado para la sesión, comprobaremos que el documento puede recuperarse y eliminaremos ese mismo ID al terminar. La eliminación del documento temporal de prueba evita que una prueba de integración altere la colección compartida con los laboratorios nativos.

También inspeccionaremos el payload resultante. La abstracción puede añadir metadata propia o adaptar el formato del documento; observarlo permite entender qué capa ha escrito realmente cada campo.

Esta prueba no comprueba lo mismo que la ingesta masiva. La ingesta verifica que sabemos cargar el catálogo; esta prueba verifica que la capa de LangChain puede escribir y borrar sin cambiar la identidad del documento ni dejar residuos en la colección.

Como utilizamos un único ID conocido, cualquier efecto de la operación se puede localizar y comprobar. Esa acotación hace que una prueba de integración sea segura y fácil de interpretar.


In [ ]:
langchain_temporary_test_id = record_id_for_product("S03-LANGCHAIN-TEMPORARY-TEST")
temporary_test_document = Document(
    id=langchain_temporary_test_id,
    page_content="taladro inalámbrico temporal de prueba para comprobar la integración",
    metadata={
        "record_id": langchain_temporary_test_id,
        "product_id": "S03-LC-TEMPORARY-TEST",
        "vector_id": -2,
        "title": "Registro temporal de prueba de LangChain",
        "brand": "S03",
        "color": "amarillo",
        "locale": "es",
    },
)
temporary_test_outcomes = {}
for provider, store in stores.items():
    returned_ids = store.add_documents([temporary_test_document], ids=[langchain_temporary_test_id])
    assert langchain_temporary_test_id in returned_ids
    visible = store.similarity_search("taladro inalámbrico temporal de prueba", k=5)
    assert any(document.metadata.get("record_id") == langchain_temporary_test_id for document in visible)
    store.delete(ids=[langchain_temporary_test_id])
    after_delete = store.similarity_search("taladro inalámbrico temporal de prueba", k=20)
    assert all(document.metadata.get("record_id") != langchain_temporary_test_id for document in after_delete)
    temporary_test_outcomes[provider] = {"inserted": True, "visible": True, "deleted": True}
temporary_test_outcomes

<a id="s03-langchain-cadena"></a>

## 9. Cadena de recuperación

Todavía no necesitamos un LLM para construir una cadena. Recuperar documentos y formatearlos como contexto ya es una composición útil y auditable: podemos ver qué fragmentos entrarán en la siguiente etapa y de qué IDs proceden.

Mantener esa trazabilidad es importante. Si más adelante un modelo generativo responde algo incorrecto, necesitaremos separar si el contexto recuperado era insuficiente, si contenía información equivocada o si el problema apareció durante la generación.

En este notebook detenemos la cadena antes del LLM precisamente para no mezclar esas capas.

El formato del contexto tampoco es inocente. Si eliminamos los IDs, el siguiente componente recibe texto limpio pero perdemos la posibilidad de atribuir una frase a una fuente. Si incluimos demasiados campos, ganamos trazabilidad pero aumentamos el tamaño y ruido del contexto.

Detenernos antes del LLM permite observar ese compromiso sin mezclarlo con la generación. Así podemos saber qué recuperamos y cómo lo presentamos antes de preguntarnos qué hará un modelo con ello.


In [ ]:
def format_context(documents):
    return "\n\n".join(
        f"[{document.metadata.get('record_id')}] {document.page_content}"
        for document in documents
    )

context_chains = {
    provider: retriever | RunnableLambda(format_context)
    for provider, retriever in retrievers.items()
}
contexts = {
    provider: chain.invoke(drill_text)
    for provider, chain in context_chains.items()
}
for provider, context in contexts.items():
    print(f"\n--- {provider.upper()} ---\n{context[:1200]}")

La cadena no comprueba que el contexto sea verdadero ni suficiente. Solo transforma una lista de documentos en una representación que otra componente puede consumir.

En una sesión posterior podríamos añadir generación, pero entonces cambiaría la pregunta y necesitaríamos evaluar la trazabilidad completa desde la respuesta hasta los IDs recuperados aquí.


<a id="s03-langchain-comparacion"></a>

## 10. Comparación con SDK nativo

La interfaz común no debería alterar silenciosamente el ranking cuando entregamos exactamente el mismo vector y no aplicamos MMR. Para comprobarlo, compararemos la búsqueda por vector de LangChain con la consulta equivalente realizada mediante el SDK nativo.

Esta prueba evita introducir otra diferencia a través del encoder. No preguntamos si dos consultas de texto producen listas parecidas; preguntamos si ambos adaptadores recuperan los mismos IDs a partir del mismo vector, colección y configuración.

Es una prueba de no regresión pequeña, pero muy útil. Si falla, sabemos que debemos inspeccionar la integración antes de confiar en la abstracción.

La comparación se hace con IDs y no solo con títulos. El ID es la evidencia más directa de que ambos caminos recuperaron el mismo registro lógico. Si los IDs coinciden pero el título no, el fallo está en la correspondencia entre identidad y metadata, no en el ranking.

También evitamos MMR y una nueva consulta de texto. De ese modo eliminamos transformaciones que podrían cambiar legítimamente el orden y concentramos la prueba en la adaptación del vector store.


In [ ]:
ranking_checks = {}
if "chroma" in stores:
    native_ids = native_clients["chroma"].query(
        query_embeddings=[reference_query.tolist()], n_results=10
    )["ids"][0]
    langchain_docs = stores["chroma"].similarity_search_by_vector(reference_query.tolist(), k=10)
    langchain_ids = [document.metadata["record_id"] for document in langchain_docs]
    ranking_checks["chroma"] = {"native": native_ids, "langchain": langchain_ids, "equal": native_ids == langchain_ids}
if "qdrant" in stores:
    native_points = native_clients["qdrant"].query_points(
        QDRANT_COLLECTION, query=reference_query.tolist(), limit=10, with_payload=True
    ).points
    native_ids = [str(point.id) for point in native_points]
    langchain_docs = stores["qdrant"].similarity_search_by_vector(reference_query.tolist(), k=10)
    langchain_ids = [document.metadata["record_id"] for document in langchain_docs]
    ranking_checks["qdrant"] = {"native": native_ids, "langchain": langchain_ids, "equal": native_ids == langchain_ids}
assert all(check["equal"] for check in ranking_checks.values()), ranking_checks
ranking_checks

La igualdad de IDs no demuestra que LangChain sea neutral en todas las operaciones. Demuestra algo más concreto: para esta búsqueda por vector, este esquema y estas versiones, el adaptador no ha alterado el ranking.

Filtros, scores, errores, namespaces, administración y consistencia siguen necesitando pruebas propias. Una abstracción se valida operación por operación, no por el simple hecho de que una consulta haya funcionado.


<a id="s03-langchain-limites"></a>

## 11. Límites de la abstracción

LangChain resulta valioso cuando queremos convertir resultados a Document, componer retrievers, intercambiar estrategias de recuperación, aplicar MMR o preparar contexto para una cadena. En esos casos, su contrato pequeño reduce trabajo repetido y hace visible el flujo de la aplicación.

El SDK nativo sigue siendo preferible para crear y configurar colecciones, administrar recursos, ajustar rendimiento, controlar batching y timeouts, diagnosticar errores, observar consistencia y aprovechar capacidades avanzadas. Es también la vía que conserva el contrato exacto del score y del filtro.

No utilizamos la integración de Pinecone en este notebook porque su dependencia actual no es compatible con la versión del SDK nativo fijada para la sesión. Forzar dos entornos incompatibles enseñaría una lección equivocada: la abstracción debe simplificar una arquitectura real, no ocultar conflictos de versiones.

> **Decisión final.** Utiliza LangChain cuando la composición que ofrece coincide con el problema. Conserva una vía nativa para administración, rendimiento, depuración y cualquier garantía que no aparezca en la interfaz del retriever.

Esta división evita dos extremos. No hace falta renunciar a LangChain para conservar control técnico, ni utilizar el SDK nativo para cada transformación de documentos. Una arquitectura razonable puede administrar y observar con el SDK nativo, y usar LangChain en el borde donde necesita componer recuperación y contexto.

La elección debe partir del requisito. Si necesitamos crear una colección, ajustar HNSW o diagnosticar consistencia, empezamos por el proveedor. Si necesitamos transportar documentos entre transformaciones, LangChain ofrece una interfaz útil.


<a id="s03-langchain-proximos-pasos"></a>

## 12. Próximos pasos

No hemos añadido LLM, agentes, LangSmith, búsqueda híbrida, vectores sparse, reranking o inferencia integrada. Cada una puede ser una evolución razonable, pero incorporarlas simultáneamente impediría saber por qué mejora o empeora un resultado.

La siguiente ampliación debería comenzar con una limitación observable. Por ejemplo: mejorar la recuperación de referencias exactas, reducir redundancia sin degradar demasiado la relevancia o aislar tenants con una garantía verificable. A partir de ahí añadiremos una capacidad cada vez y repetiremos la evaluación.

El orden de las extensiones importa. Añadir LLM y reranking a la vez podría mejorar una demo, pero impediría saber qué componente introdujo la mejora, cuál añadió coste y cuál degradó la trazabilidad.

El mismo principio que usamos al comparar bases vectoriales se aplica aquí: una hipótesis, una modificación y una medida que permita evaluar su efecto.
